# Tennis Sensor Analysis: Cloud Parallel Processing

This notebook processes raw EMG/IMU logs from the ESP32 Tennis Sensor system.

It uses **parallel processing** (when available) to analyze multiple sessions simultaneously, extracting fatigue features and swing biomechanics.

### Prerequisites:

1. **Google Drive** mounted (or local Jupyter with files accessible)
2. **Python Scripts** (`emg_util.py`, `emg_util_yaml_runner.py`, `capture_loader.py`, `capture_health.py`, `imu_feature.py`) uploaded to a folder in Drive
3. **Log Files** - Either:
   - Upload ZIP file to Drive (easier!)
   - Upload individual CSV files (`emg_log_*.csv`, `imu_log_*.csv`) to Drive

### What this does:
- Discovers all sessions in your log directory
- Runs the same analysis as `emg_util_yaml_runner.py` but faster (parallel on cloud)
- Aggregates results into a master training dataset
- Visualizes fatigue progression across all sessions
- **Full mode analysis** with per-swing EMG features and automatic health filtering

In [ ]:
# --- CELL 1: SETUP & MOUNT DRIVE ---
import sys
import os

# Detect environment
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
    print('✅ Google Drive mounted.')
except ImportError:
    IN_COLAB = False
    print('ℹ️  Running locally (not in Colab).')

# Install dependencies
if IN_COLAB:
    !pip install -q pyyaml scipy pandas matplotlib seaborn scikit-learn
else:
    print('ℹ️  Assuming dependencies already installed locally.')

print('✅ Setup complete.')

In [ ]:
# --- CELL 2: UNZIP DATA (Optional - skip if using individual CSVs) ---
import zipfile

# ========================================================================
# OPTION 1: If you uploaded a ZIP file, configure these paths:
# ========================================================================
USE_ZIP = True  # Set to False if you uploaded individual CSV files

if IN_COLAB:
    ZIP_PATH = '/content/drive/MyDrive/Tennis_Project/logs_new.zip'  # ← Update this
    EXTRACT_TO = '/content/drive/MyDrive/Tennis_Project/raw_logs'    # ← Where to extract
else:
    # Local paths (if running locally with a ZIP)
    ZIP_PATH = r'C:/wangx/ftc/arduino/py_scripts/logs_new.zip'
    EXTRACT_TO = r'C:/wangx/ftc/arduino/py_scripts/logs_new'

# Extract ZIP if enabled
if USE_ZIP:
    if os.path.exists(ZIP_PATH):
        print(f'📦 Found ZIP: {ZIP_PATH}')
        print(f'📂 Extracting to: {EXTRACT_TO}...')
        
        os.makedirs(EXTRACT_TO, exist_ok=True)
        
        with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
            zip_ref.extractall(EXTRACT_TO)
        
        # Count extracted files
        all_files = os.listdir(EXTRACT_TO)
        n_emg = len([f for f in all_files if f.startswith('emg_log_') and f.endswith('.csv')])
        n_imu = len([f for f in all_files if f.startswith('imu_log_') and f.endswith('.csv')])
        print(f'✅ Extracted {n_emg} EMG files, {n_imu} IMU files')
    else:
        print(f'⚠️  ZIP not found: {ZIP_PATH}')
        print('   Please update ZIP_PATH above or set USE_ZIP=False')
else:
    print('ℹ️  ZIP extraction disabled (USE_ZIP=False)')
    print('   Assuming CSV files are already in place.')

In [ ]:
# --- CELL 3: PATH CONFIGURATION ---
# Update these paths to match your setup

if IN_COLAB:
    # === COLAB PATHS (update to match your Drive layout) ===
    SCRIPT_DIR = '/content/drive/MyDrive/Tennis_Project/scripts'
    
    # If you used ZIP extraction above, LOG_DIR should match EXTRACT_TO
    LOG_DIR = '/content/drive/MyDrive/Tennis_Project/raw_logs'
    OUT_DIR = '/content/drive/MyDrive/Tennis_Project/processed_out'
else:
    # === LOCAL PATHS (update to match your PC) ===
    SCRIPT_DIR = r'C:/wangx/ftc/emg_util'  # where .py files are
    LOG_DIR = r'C:/wangx/ftc/arduino/py_scripts/logs_new'  # where CSVs are
    OUT_DIR = r'C:/wangx/ftc/emg_util/out'  # where results go

os.makedirs(OUT_DIR, exist_ok=True)

# Add scripts to path
if os.path.exists(SCRIPT_DIR):
    if SCRIPT_DIR not in sys.path:
        sys.path.insert(0, SCRIPT_DIR)
    print(f'✅ Script dir: {SCRIPT_DIR}')
else:
    print(f'⚠️  Script dir NOT found: {SCRIPT_DIR}')
    print('   Update SCRIPT_DIR above to point to your .py files.')

if os.path.exists(LOG_DIR):
    n_emg = len([f for f in os.listdir(LOG_DIR) if f.startswith('emg_log_') and f.endswith('.csv')])
    n_imu = len([f for f in os.listdir(LOG_DIR) if f.startswith('imu_log_') and f.endswith('.csv')])
    print(f'✅ Log dir: {LOG_DIR}  ({n_emg} EMG, {n_imu} IMU files)')
else:
    print(f'⚠️  Log dir NOT found: {LOG_DIR}')
    print('   Update LOG_DIR above to point to your log files.')

In [ ]:
# --- CELL 4: IMPORTS ---
import json
import traceback
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import emg_util_yaml_runner as runner
    import capture_loader as loader
    import emg_util as core
    import capture_health as health
    print('✅ Tennis analysis modules imported.')
except ImportError as e:
    print(f'❌ Import error: {e}')
    print('   Check that SCRIPT_DIR is correct and all .py files are present.')
    raise

In [ ]:
# --- CELL 5: CONFIGURATION ---
# This mirrors config.yaml. Adjust any parameter here.
# This is the SAME config used by emg_util_yaml_runner.py locally.
# NO config.yaml FILE NEEDED for the notebook!

CLOUD_CONFIG = {
    'session': {
        'emg_glob': 'emg_log_*.csv',
        'imu_glob': 'imu_log_*.csv',
        'vitals_path': 'vitals_log.csv',
        'emg_fs_hz': 2000,
        'imu_fs_fallback_hz': 200,
        'use_physical_units': True,
        'acc_fsr_default_g': 2,
        'gyro_fsr_default_dps': 250,
        'out_dir': OUT_DIR,
    },
    'features': {
        'mode': 'both',  # 'bout' (fast) | 'both' (FULL per-swing EMG) ← DEFAULT
        'compute_windowed': False,
        'zc_thr': 0.01,
        'ssc_thr': 0.01,
    },
    'swing_detection': {
        'use_gyro': True,
        'baseline_s': 2.0,
        'z_thr': 5.0,
        'min_consec': 3,
        'min_gap_ms': 300.0,
        'pre_ms': 250.0,
        'post_ms': 450.0,
        'mdf_pad_ms': 300.0,
    },
    'baseline': {
        'mode': 'swing',
        'warmup_skip_min': 5.0,
        'warmup_skip_swings': 0,
        'baseline_swings': 20,
        'baseline_window_sec': 60.0,
    },
    'bout': {
        'gap_thr_s': 10,
        'gap_thr_long_s': 30,
        'min_bout_swings': 3,
        'rest_dur_s': 4,
        'merge_short_rest_s': 6,
    },
    'plots': {
        'enable': True,
        'full_duration': True,
        'emg_max_points': 300000,
        'do_fatigue': True,
        'win_sec': 1.0,
        'step_sec': 0.25,
    },
    'diagnostics': {
        'emg_imu_correlation': {'enable': True, 'method': 'pearson'},
        'activity_overlay': {'enable': True, 'plot_swings': True},
    },
    'activity_gating': {
        'enable': True,
        'w_gyro': 0.8,
        'w_acc': 0.2,
        'smooth_ms': 250.0,
        'p_high': 90,
        'p_low': 75,
        'filter_swings': False,
    },
    'health': {
        'enable': True,  # ← EMG health filtering ENABLED (automatic)
        'filter_swings_by_emg_bad': True,
    },
}

print('✅ Configuration loaded.')
print(f'   Mode: {CLOUD_CONFIG["features"]["mode"]} (full per-swing analysis)')
print(f'   EMG fs: {CLOUD_CONFIG["session"]["emg_fs_hz"]} Hz')
print(f'   EMG health filtering: {"ENABLED" if CLOUD_CONFIG["health"]["enable"] else "DISABLED"}')
print(f'   Output: {OUT_DIR}')

In [ ]:
# --- CELL 6: SESSION DISCOVERY ---
if not os.path.exists(LOG_DIR):
    raise FileNotFoundError(f'Log directory not found: {LOG_DIR}')

print(f'Scanning: {LOG_DIR}')
sessions = loader.discover_sessions(
    LOG_DIR,
    emg_glob=CLOUD_CONFIG['session']['emg_glob'],
    imu_glob=CLOUD_CONFIG['session']['imu_glob'],
    vitals_name=CLOUD_CONFIG['session']['vitals_path'],
)

session_ids = sorted(sessions.keys())
print(f'\nFound {len(session_ids)} session(s): {session_ids}')

if not session_ids:
    raise SystemExit('No sessions found. Check LOG_DIR and file naming.')

In [ ]:
# --- CELL 7: RUN ANALYSIS ---
# Sequential by default (safe for Colab). Set USE_PARALLEL=True for local Jupyter with many sessions.

USE_PARALLEL = False  # Set True for local Jupyter with many sessions

results = []  # list of (session_id, status, out_path_or_error)

def process_one(session_id):
    """Process a single session using analyze_single_session from emg_util_yaml_runner.py"""
    files = sessions[session_id]
    try:
        out_path = runner.analyze_single_session(
            session_id, files, CLOUD_CONFIG, OUT_DIR
        )
        return (session_id, 'SUCCESS', out_path)
    except Exception as e:
        tb = traceback.format_exc()
        return (session_id, 'FAILED', tb)

if USE_PARALLEL:
    import multiprocessing
    from concurrent.futures import ProcessPoolExecutor
    n_workers = multiprocessing.cpu_count()
    print(f'Running parallel on {n_workers} cores...')
    with ProcessPoolExecutor(max_workers=n_workers) as ex:
        for res in ex.map(process_one, session_ids):
            results.append(res)
            print(f'[{res[1]}] {res[0]}')
            if res[1] == 'FAILED':
                print(res[2][:500])
else:
    print(f'Running sequentially ({len(session_ids)} session(s))...')
    for i, sid in enumerate(session_ids, 1):
        print(f'\n[{i}/{len(session_ids)}] {sid}')
        res = process_one(sid)
        results.append(res)
        if res[1] == 'FAILED':
            print(f'  ❌ FAILED:\n{res[2][:800]}')
        else:
            print(f'  ✅ Done → {res[2]}')

n_ok = sum(1 for r in results if r[1] == 'SUCCESS')
n_fail = sum(1 for r in results if r[1] == 'FAILED')
print(f'\n=== Batch complete: {n_ok} succeeded, {n_fail} failed ===')

In [ ]:
# --- CELL 8: AGGREGATE RESULTS ---
all_swings = []
all_bouts = []

for sid, status, out_or_err in results:
    if status != 'SUCCESS':
        continue
    out_dir = out_or_err
    
    # Load swing features
    sf_path = os.path.join(out_dir, 'swing_features.csv')
    if os.path.exists(sf_path):
        try:
            df = pd.read_csv(sf_path)
            df['session_id'] = sid
            all_swings.append(df)
        except Exception as e:
            print(f'Warning: could not read {sf_path}: {e}')
    
    # Load bout summary
    bout_path = os.path.join(out_dir, 'bout_summary.csv')
    if os.path.exists(bout_path):
        try:
            df = pd.read_csv(bout_path)
            df['session_id'] = sid
            all_bouts.append(df)
        except Exception as e:
            print(f'Warning: could not read {bout_path}: {e}')

MASTER_DF = pd.concat(all_swings, ignore_index=True) if all_swings else pd.DataFrame()
BOUTS_DF = pd.concat(all_bouts, ignore_index=True) if all_bouts else pd.DataFrame()

print(f'Master swing dataset: {len(MASTER_DF)} swings from {len(all_swings)} sessions')
print(f'Bouts dataset: {len(BOUTS_DF)} bouts')

if len(MASTER_DF):
    save_path = os.path.join(OUT_DIR, 'MASTER_TRAINING_DATA.csv')
    MASTER_DF.to_csv(save_path, index=False)
    print(f'\n✅ Saved: {save_path}')
    print(f'Columns: {list(MASTER_DF.columns)}')
    print(MASTER_DF.describe().round(3))

In [ ]:
# --- CELL 9: VISUALIZE FATIGUE PROGRESSION ---
if len(MASTER_DF) == 0:
    print('No swing data available. Run previous cells first.')
else:
    df = MASTER_DF.copy()
    df['t_min'] = df['t_start_ms'] / 60000.0
    df['cost_index'] = df['rms'] / df['imu_peak_mag'].replace(0, np.nan).fillna(1)
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('Fatigue Analysis — All Sessions', fontsize=14, fontweight='bold')
    
    # 1. MDF over time
    ax = axes[0, 0]
    for sid in df['session_id'].unique():
        sub = df[df['session_id'] == sid].sort_values('t_min')
        ax.plot(sub['t_min'], sub['mdf'], alpha=0.7, lw=1.2, label=sid)
    ax.set_xlabel('Time (min)'); ax.set_ylabel('MDF (Hz)')
    ax.set_title('Median Frequency (↓ = fatigue)')
    ax.legend(fontsize=7); ax.grid(True, alpha=0.3)
    
    # 2. RMS over time
    ax = axes[0, 1]
    for sid in df['session_id'].unique():
        sub = df[df['session_id'] == sid].sort_values('t_min')
        ax.plot(sub['t_min'], sub['rms'], alpha=0.7, lw=1.2, label=sid)
    ax.set_xlabel('Time (min)'); ax.set_ylabel('RMS')
    ax.set_title('EMG RMS (↑ = more effort)')
    ax.legend(fontsize=7); ax.grid(True, alpha=0.3)
    
    # 3. Cost Index scatter
    ax = axes[1, 0]
    sc = ax.scatter(df['t_min'], df['cost_index'],
                    c=df['mdf'], cmap='RdYlGn', alpha=0.6, s=20)
    plt.colorbar(sc, ax=ax, label='MDF (Hz)')
    ax.set_xlabel('Time (min)'); ax.set_ylabel('Cost Index (RMS / SwingSpeed)')
    ax.set_title('Neuromechanical Cost (color=MDF)')
    ax.grid(True, alpha=0.3)
    
    # 4. MDF drop % (if available)
    ax = axes[1, 1]
    mdf_col = 'swing_mdf_drop_pct' if 'swing_mdf_drop_pct' in df.columns else 'mdf_drop_pct'
    if mdf_col in df.columns:
        for sid in df['session_id'].unique():
            sub = df[df['session_id'] == sid].sort_values('t_min')
            ax.plot(sub['t_min'], sub[mdf_col], alpha=0.7, lw=1.2, label=sid)
        ax.axhline(12, color='r', ls='--', lw=1, label='12% threshold')
        ax.set_xlabel('Time (min)'); ax.set_ylabel('MDF Drop (%)')
        ax.set_title('MDF Drop % from Baseline')
        ax.legend(fontsize=7); ax.grid(True, alpha=0.3)
    else:
        ax.text(0.5, 0.5, 'mdf_drop_pct not available', ha='center', va='center', transform=ax.transAxes)
    
    plt.tight_layout()
    dash_path = os.path.join(OUT_DIR, 'dashboard_fatigue.png')
    fig.savefig(dash_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'✅ Saved: {dash_path}')